# Fill missing focus scores with respondent preference models

This notebook trains one model per `respondentId` from `data/raw/4_comfort_perception.csv`, joined to nearby room measurements, then randomly selects among those respondent models to fill `focus_score` for every row in the large unified environmental dataset. The filled target is on the 1-5 comfort/focus scale.

In [1]:
from pathlib import Path
import sys

import pandas as pd

MAL_DIR = Path.cwd()
if MAL_DIR.name != "MAL":
    MAL_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "MAL")

sys.path.insert(0, str(MAL_DIR))

from scripts.build_unified_environment_dataset import build_unified_dataset
from scripts.fill_missing_targets import (
    build_respondent_comfort_training_data,
    fill_focus_scores_from_respondent_preferences,
)

PROCESSED_DIR = MAL_DIR / "data" / "processed"
UNIFIED_PATH = PROCESSED_DIR / "final_mock_dataset.csv"
REPORT_PATH = PROCESSED_DIR / "unified_environment_focus_dataset_report.csv"
ROOM_MEASUREMENTS_PATH = MAL_DIR / "data" / "raw" / "data_8" / "smart-campus-comfort-data" / "1_room_measurements.csv"
COMFORT_PERCEPTION_PATH = MAL_DIR / "data" / "raw" / "data_8" / "smart-campus-comfort-data" / "4_comfort_perception.csv"
RESPONDENT_TRAINING_PATH = PROCESSED_DIR / "respondent_comfort_training_dataset.csv"
FILLED_FOCUS_PATH = PROCESSED_DIR / "unified_environment_respondent_focus_score_filled.csv"

UNIFIED_PATH, COMFORT_PERCEPTION_PATH, FILLED_FOCUS_PATH

(WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/final_mock_dataset.csv'),
 WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/raw/data_8/smart-campus-comfort-data/4_comfort_perception.csv'),
 WindowsPath('c:/Users/piotr/Desktop/University/Semester4/SEP4/SEP4/MAL/data/processed/unified_environment_respondent_focus_score_filled.csv'))

## Load environmental rows and respondent comfort data

In [2]:
if not UNIFIED_PATH.exists():
    unified_df, source_report = build_unified_dataset(
        output_path=UNIFIED_PATH,
        report_path=REPORT_PATH,
        allow_missing=False,
    )
else:
    unified_df = pd.read_csv(UNIFIED_PATH, low_memory=False)

measurements_df = pd.read_csv(ROOM_MEASUREMENTS_PATH, low_memory=False)
comfort_df = pd.read_csv(COMFORT_PERCEPTION_PATH, low_memory=False)

input_summary = pd.DataFrame([
    {
        "environment_rows_to_fill": len(unified_df),
        "comfort_response_rows": len(comfort_df),
        "respondent_models_available": int(comfort_df["respondentId"].nunique()),
        "comfort_value_classes": sorted(comfort_df["comfortValue"].dropna().astype(int).unique().tolist()),
    }
])
input_summary

,environment_rows_to_fill,comfort_response_rows,respondent_models_available,comfort_value_classes
0,947898,1830,182,"[1, 2, 3, 4, 5]"


## Build respondent training rows

In [3]:
respondent_training_df = build_respondent_comfort_training_data(measurements_df, comfort_df)
respondent_training_df.to_csv(RESPONDENT_TRAINING_PATH, index=False)

respondent_training_summary = pd.DataFrame([
    {
        "training_rows": len(respondent_training_df),
        "respondents": int(respondent_training_df["respondent_id"].nunique()),
        "missing_focus_score": int(respondent_training_df["focus_score"].isna().sum()),
    }
])
respondent_training_summary

,training_rows,respondents,missing_focus_score
0,1830,182,0


In [4]:
respondent_training_df.groupby("respondent_id")["focus_score"].agg(
    training_rows="size",
    score_classes="nunique",
    mean_score="mean",
).describe()

,training_rows,score_classes,mean_score
count,182.000000,182.000000,182.000000
mean,10.054945,2.445055,3.361490
std,6.525534,1.153784,0.584953
min,1.000000,1.000000,2.000000
25%,3.000000,1.000000,3.000000
50%,10.500000,2.000000,3.250000
75%,16.000000,3.000000,3.774510
max,21.000000,5.000000,5.000000


## Train one model per respondent and fill 1-5 focus_score

The shared environmental features are `humidity`, `temperature`, `noise`, and `co2`. Rows that already had `focus_score` keep their original value; missing rows are filled by randomly selecting one of the respondent models.

In [5]:
filled_df, model_summary = fill_focus_scores_from_respondent_preferences(
    unified_df,
    respondent_training_df,
    feature_columns=["humidity", "temperature", "noise", "co2"],
    random_state=42,
)

filled_df.to_csv(FILLED_FOCUS_PATH, index=False)
model_summary

,student_id,training_rows,classes
0,1,1,4
1,2,9,"2,3,4,5"
2,3,8,"3,4"
3,4,19,"2,3,4,5"
4,5,12,"2,3"
...,...,...,...
177,182,1,3
178,184,12,"3,4"
179,185,14,"2,3,4"
180,187,2,3


## Verify every row has a 1-5 target

In [6]:
target_check = pd.DataFrame([
    {
        "output_csv": str(FILLED_FOCUS_PATH),
        "rows": len(filled_df),
        "rows_with_focus_score": int(filled_df["focus_score"].notna().sum()),
        "rows_missing_focus_score": int(filled_df["focus_score"].isna().sum()),
        "all_rows_have_focus_score": bool(filled_df["focus_score"].notna().all()),
        "respondent_models_trained": len(model_summary),
        "min_focus_score": int(filled_df["focus_score"].min()),
        "max_focus_score": int(filled_df["focus_score"].max()),
    }
])
target_check

,output_csv,rows,rows_with_focus_score,rows_missing_focus_score,all_rows_have_focus_score,respondent_models_trained,min_focus_score,max_focus_score
0,c:\Users\piotr\Desktop\University\Semester4\SE...,947898,947898,0,True,182,1,5


In [7]:
assert filled_df["focus_score"].notna().all(), "Some rows still have missing focus_score"
assert filled_df["focus_score"].between(1, 5).all(), "focus_score should be on the 1-5 scale"
print("All rows have 1-5 focus_score values filled by randomly selected respondent models.")

All rows have 1-5 focus_score values filled by randomly selected respondent models.


## Focus score distribution

In [8]:
focus_score_distribution = (
    filled_df["focus_score"]
    .value_counts()
    .sort_index()
    .rename_axis("focus_score")
    .reset_index(name="rows")
)
focus_score_distribution

,focus_score,rows
0,1,13312
1,2,110177
2,3,450218
3,4,248581
4,5,125610


In [9]:
pd.crosstab(filled_df["source"], filled_df["focus_score"])

focus_score,1,2,3,4,5
source,,,,,
homecoach_5min_2023,1472,10117,36190,19249,10017
homecoach_5min_2024,1463,12166,48171,25435,13090
homecoach_5min_2025,1517,12252,47869,25428,13124
homecoach_5min_2026,241,3720,16869,8776,4526
keti_1min_resampled,7285,59130,247259,140756,70209
room_conditions,25,316,1275,701,348
room_measurements,1309,12476,52585,28236,14296


## Preview of the saved filled dataset

In [10]:
filled_df.head(20)

,timestamp,session_id,location_id,record_id,source,humidity,light,temperature,noise,co2,focus_score
0,2013-08-23T23:04:00Z,keti_1min_resampled__510__s00002,510,keti_1min_resampled:row_00192747,keti_1min_resampled,2768.864400,5.323010,23.460000,7.082283,0.002475,4
1,2013-08-23T23:04:00Z,keti_1min_resampled__511__s00001,511,keti_1min_resampled:row_00205226,keti_1min_resampled,2782.562500,5.527443,22.620000,7.082266,0.002571,2
2,2013-08-23T23:04:00Z,keti_1min_resampled__558__s00001,558,keti_1min_resampled:row_00257060,keti_1min_resampled,2859.040900,1.386294,23.120000,5.690815,0.002364,5
3,2013-08-23T23:04:00Z,keti_1min_resampled__621__s00001,621,keti_1min_resampled:row_00293044,keti_1min_resampled,2412.283225,4.226834,25.590000,5.957954,0.002000,3
4,2013-08-23T23:04:00Z,keti_1min_resampled__621a__s00001,621a,keti_1min_resampled:row_00305039,keti_1min_resampled,3050.352900,4.840242,22.210000,6.930215,0.002597,3
5,2013-08-23T23:04:00Z,keti_1min_resampled__621c__s00001,621c,keti_1min_resampled:row_00317034,keti_1min_resampled,2403.940900,5.598422,24.505000,7.067293,0.001880,2
6,2013-08-23T23:04:00Z,keti_1min_resampled__640__s00001,640,keti_1min_resampled:row_00352658,keti_1min_resampled,2753.625625,2.197225,22.740000,5.779496,0.002198,3
7,2013-08-23T23:04:00Z,keti_1min_resampled__644__s00001,644,keti_1min_resampled:row_00364653,keti_1min_resampled,2744.188225,5.111988,22.805000,6.958249,0.002134,3
8,2013-08-23T23:04:00Z,keti_1min_resampled__656a__s00001,656a,keti_1min_resampled:row_00388863,keti_1min_resampled,2490.010000,5.178971,24.370000,6.705446,0.001729,4
9,2013-08-23T23:04:00Z,keti_1min_resampled__717__s00001,717,keti_1min_resampled:row_00448834,keti_1min_resampled,2623.488400,4.744932,23.340000,6.289319,0.002475,4
